# Q3: Data Wrangling

**Phase 4:** Data Wrangling & Transformation  

In [124]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load cleaned data from Q2
df = pd.read_csv('output/q2_cleaned_data.csv')
print(f"Loaded {len(df):,} cleaned records")

# Display first few rows
print("\nFirst few rows:")
print(df.head())

Loaded 195,892 cleaned records

First few rows:
                  Station Name Measurement Timestamp  Air Temperature  \
0  63rd Street Weather Station   2015-04-25 09:00:00             7.00   
1  63rd Street Weather Station   2015-04-30 05:00:00             6.10   
2   Oak Street Weather Station   2015-05-22 15:00:00             6.10   
3       Foster Weather Station   2015-05-22 16:00:00             9.17   
4       Foster Weather Station   2015-05-22 17:00:00             9.28   

   Wet Bulb Temperature  Humidity  Rain Intensity  Interval Rain  Total Rain  \
0                   5.9        86             0.0            0.0         5.2   
1                   4.3        76             0.0            0.0         2.5   
2                   7.0        55             0.0            0.0         1.4   
3                   7.0        59             0.0            0.0         1.4   
4                   7.0        61             0.0            0.0         1.4   

   Precipitation Type  Wind Dire

## 

## Parse DataTime

In [125]:
# Identify datetime column
datetime_col = "Measurement Timestamp"
df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce")

# Drop rows where datetime is missing (failed parsing)
before = len(df)
df = df.dropna(subset=[datetime_col])
after = len(df)
print(f"Dropped {before - after} rows with invalid datetimes")

# Set datetime index and sort
df = df.set_index(datetime_col) # set datetime as index
df = df.sort_index() #ensures dasta is chronologically ordered

Dropped 0 rows with invalid datetimes


## Extract Temporal Features

In [126]:
# Extract hour, day of week, month, year, day name, is weekend from Measurement Timestamp column

df['hour'] = df.index.hour
print("  ✓ Extracted: hour (0-23)")

df['day_of_week'] = df.index.dayofweek  # 0=Monday, 6=Sunday
print("  ✓ Extracted: day_of_week (0=Monday, 6=Sunday)")

df['month'] = df.index.month
print("  ✓ Extracted: month (1-12)")

df['year'] = df.index.year
print("  ✓ Extracted: year")

df['day_name'] = df.index.day_name()
print("  ✓ Extracted: day_name (e.g., 'Monday')")

df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)  # 1 if Sat/Sun, 0 otherwise
print("  ✓ Extracted: is_weekend (0 or 1)")

df['day_of_month'] = df.index.day
print("  ✓ Extracted: day_of_month (1-31)")

df['quarter'] = df.index.quarter
print("  ✓ Extracted: quarter (1-4)")

# Verify extraction
print("\nTemporal features summary:")
temporal_features = ['hour', 'day_of_week', 'month', 'year', 'day_name', 'is_weekend']
print(df[temporal_features].head(5))

  ✓ Extracted: hour (0-23)
  ✓ Extracted: day_of_week (0=Monday, 6=Sunday)
  ✓ Extracted: month (1-12)
  ✓ Extracted: year
  ✓ Extracted: day_name (e.g., 'Monday')
  ✓ Extracted: is_weekend (0 or 1)
  ✓ Extracted: day_of_month (1-31)
  ✓ Extracted: quarter (1-4)

Temporal features summary:
                       hour  day_of_week  month  year  day_name  is_weekend
Measurement Timestamp                                                      
2015-04-25 09:00:00       9            5      4  2015  Saturday           1
2015-04-30 05:00:00       5            3      4  2015  Thursday           0
2015-05-22 15:00:00      15            4      5  2015    Friday           0
2015-05-22 16:00:00      16            4      5  2015    Friday           0
2015-05-22 17:00:00      17            4      5  2015    Friday           0


## Calculate Date Range

In [127]:
# Get start and end dates, duration, total days, total records
start_date = df.index.min()
end_date = df.index.max()
duration = end_date - start_date
years = duration.days // 365
months = (duration.days % 365) // 30
days = (duration.days % 365) % 30
hours = duration.seconds // 3600

print(f"\nDate Range Information:")
print(f"  Start: {start_date}")
print(f"  End: {end_date}")
print(f"  Total Duration: {years} years, {months} months, {days} days, {hours} hours")
print(f"  Total Records: {len(df):,}")


Date Range Information:
  Start: 2015-04-25 09:00:00
  End: 2025-11-24 12:00:00
  Total Duration: 10 years, 7 months, 6 days, 3 hours
  Total Records: 195,892


## Saving Artifacts

In [128]:
# SAVE ARTIFACT 1: q3_wrangled_data.csv
# Reset index to save datetime as a column 
print("\nSaving wrangled data...")
df_to_save = df.reset_index()
df_to_save.to_csv('output/q3_wrangled_data.csv', index=False)
print(f"  Columns saved: {df_to_save.shape[1]}")
print(f"  Rows saved: {df_to_save.shape[0]}")
print("✓ Saved: output/q3_wrangled_data.csv")

# SAVE ARTIFACT 2: q3_temporal_features.csv
print("\nSaving temporal features...")

# Select temporal features to save (required + optional)
temporal_cols = ['hour', 'day_of_week', 'month', 'year', 'day_name', 'is_weekend']
# Reset index to include datetime as a column
temporal_df = df[temporal_cols].reset_index()
temporal_df.to_csv('output/q3_temporal_features.csv', index=False)
print(f"  Temporal features: {', '.join(temporal_cols)}")
# Display sample of temporal features
print("\nSample of temporal features saved:")
print(temporal_df.head())
print("✓ Saved: output/q3_temporal_features.csv")

# SAVE ARTIFACT 3: q3_datetime_info.txt
print("\nSaving datetime information...")

with open('output/q3_datetime_info.txt', 'w') as f:
    f.write("Date Range After Datetime Parsing:\n")
    f.write(f"Start: {start_date}\n")
    f.write(f"End: {end_date}\n")
    f.write(f"Total Duration: {years} years, {months} months, {days} days, {hours} hours\n")
    f.write(f"\nAdditional Information:\n")
    f.write(f"Total Days: {duration.days}\n")
    f.write(f"Total Records: {len(df):,}\n")
    f.write(f"Datetime Column: {datetime_col}\n")

print("✓ Saved: output/q3_datetime_info.txt")


Saving wrangled data...
  Columns saved: 26
  Rows saved: 195892
✓ Saved: output/q3_wrangled_data.csv

Saving temporal features...
  Temporal features: hour, day_of_week, month, year, day_name, is_weekend

Sample of temporal features saved:
  Measurement Timestamp  hour  day_of_week  month  year  day_name  is_weekend
0   2015-04-25 09:00:00     9            5      4  2015  Saturday           1
1   2015-04-30 05:00:00     5            3      4  2015  Thursday           0
2   2015-05-22 15:00:00    15            4      5  2015    Friday           0
3   2015-05-22 16:00:00    16            4      5  2015    Friday           0
4   2015-05-22 17:00:00    17            4      5  2015    Friday           0
✓ Saved: output/q3_temporal_features.csv

Saving datetime information...
✓ Saved: output/q3_datetime_info.txt
